In [1]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_classif
import joblib

In [ ]:
RAW_CSV_PATH = "X:/M.E/Sem1/ADM/ADM/preprocessing/MachineLearningCVE.csv"        # Adjust to your folder with raw CSVs
OUTPUT_DIR = "./curated_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_FEATURE_COUNT = 37  # 37 continuous flow features + 1 Label = 38 columns
SEED = 42

In [ ]:
csv_files = glob.glob(RAW_CSV_PATH)
if not csv_files:
    raise FileNotFoundError(f"No CSV files found matching pattern: {RAW_CSV_PATH}")

print(f"[*] Found {len(csv_files)} CSV files. Commencing ingestion...")
df_list = []

for f in sorted(csv_files):
    print(f"    -> Reading {os.path.basename(f)}...")
    temp_df = pd.read_csv(f, low_memory=False)
    # Strip accidental leading/trailing whitespaces in column headers
    temp_df.columns = temp_df.columns.str.strip()
    df_list.append(temp_df)

df = pd.concat(df_list, axis=0, ignore_index=True)
print(f"[*] Combined Raw Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

label_col = [col for col in df.columns if col.lower() == 'label'][0]
df.rename(columns={label_col: 'Label'}, inplace=True)

[*] Found 1 CSV files. Commencing ingestion...
    -> Reading MachineLearningCVE.csv...
[*] Combined Raw Shape: 2830743 rows, 79 columns.


In [ ]:
# 
# 3. Target Binarization: Benign (0) vs. Malign (1)
# 
print("[*] Binarizing Target Column: 'BENIGN' -> 0, All Attacks -> 1...")
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = (df['Label'] != 'BENIGN').astype(np.int8)

counts = df['Label'].value_counts()
print(f"    -> Benign (0): {counts.get(0, 0)} | Malign (1): {counts.get(1, 0)}")

[*] Binarizing Target Column: 'BENIGN' -> 0, All Attacks -> 1...
    -> Benign (0): 2273097 | Malign (1): 557646


In [ ]:
# 4. Drop Leakage / Identifier Columns & Handle Non-Finite Values
# Drop identity fields to avoid shortcut learning (IPs, Ports, Timestamps)
metadata_cols = [
    'Flow ID', 'Source IP', 'Destination IP', 'Source Port',
    'Destination Port', 'Timestamp', 'External IP'
]
cols_to_drop = [c for c in metadata_cols if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f"[*] Dropped metadata/identifier columns: {cols_to_drop}")

# Clean non-finite values produced by packet capture tools
print("[*] Sanitizing non-finite values (Inf, -Inf, NaN)...")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(axis=0, how='any', inplace=True)

# Remove duplicate flows
init_len = len(df)
df.drop_duplicates(inplace=True)
print(f"[*] Dropped {init_len - len(df)} duplicate/corrupt rows. Remaining rows: {len(df)}")

[*] Dropped metadata/identifier columns: ['Destination Port']
[*] Sanitizing non-finite values (Inf, -Inf, NaN)...
[*] Dropped 594712 duplicate/corrupt rows. Remaining rows: 2233164


In [ ]:
# 5. Feature Selection: Variance Threshold + Mutual Information (Entropy)
print("[*] Starting feature selection for top 37 behavioral features...")
X = df.drop(columns=['Label'])
y = df['Label']

# Step A: Filter constant or near-zero variance features
variance_mask = X.var(axis=0, numeric_only=True) > 1e-6
valid_cols = variance_mask[variance_mask].index.tolist()
X = X[valid_cols]
print(f"    -> Retained {len(valid_cols)} features after zero-variance filtering.")

# Step B: Stratified subsample to compute mutual information efficiently
sample_size = min(100000, len(X))
X_sub, _, y_sub, _ = train_test_split(
    X, y, train_size=sample_size, stratify=y, random_state=SEED
)

print(f"    -> Estimating Mutual Information on {sample_size} stratified samples...")
mi_scores = mutual_info_classif(X_sub, y_sub, random_state=SEED)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# Retain top 37 features
selected_37_features = mi_series.head(TARGET_FEATURE_COUNT).index.tolist()
print("\n[✔] Selected Top 37 Behavioral Features:")
for rank, feat in enumerate(selected_37_features, 1):
    print(f"    {rank:02d}. {feat} (MI Score: {mi_series[feat]:.4f})")

[*] Starting feature selection for top 37 behavioral features...
    -> Retained 69 features after zero-variance filtering.
    -> Estimating Mutual Information on 100000 stratified samples...

[✔] Selected Top 37 Behavioral Features:
    01. Avg Bwd Segment Size (MI Score: 0.3241)
    02. Bwd Packet Length Mean (MI Score: 0.3240)
    03. Average Packet Size (MI Score: 0.3225)
    04. Total Length of Bwd Packets (MI Score: 0.3192)
    05. Subflow Bwd Bytes (MI Score: 0.3187)
    06. Packet Length Variance (MI Score: 0.2985)
    07. Packet Length Std (MI Score: 0.2979)
    08. Bwd Packet Length Max (MI Score: 0.2925)
    09. Packet Length Mean (MI Score: 0.2843)
    10. Max Packet Length (MI Score: 0.2803)
    11. Bwd Packet Length Std (MI Score: 0.2780)
    12. Fwd IAT Max (MI Score: 0.2740)
    13. Init_Win_bytes_backward (MI Score: 0.2736)
    14. Fwd Packet Length Max (MI Score: 0.2613)
    15. Subflow Fwd Bytes (MI Score: 0.2594)
    16. Total Length of Fwd Packets (MI Score: 0.258

In [ ]:
# # 6. Assemble & Save Curated Dataset (37 Features + 1 Label = 38 Columns)
final_df = df[selected_37_features + ['Label']].copy()
print(f"\n[*] Final Dataset Matrix Dimensions: {final_df.shape} (37 Features + 1 Target Label)")



csv_out = os.path.join(OUTPUT_DIR, "cicids2017_curated_38cols.csv")
final_df.to_csv(csv_out, index=False)
print(f"[✔] Saved CSV to: {csv_out}")



[*] Final Dataset Matrix Dimensions: (2233164, 38) (37 Features + 1 Target Label)
[✔] Saved CSV to: ./curated_dataset\cicids2017_curated_38cols.csv
